# Week 1 — Checkpoint 3: Outlier Detection & Processing
### Dataset: `house_sale.csv` (bina.az — real estate sale listings)

https://www.kaggle.com/datasets/sehriyarmemmedli/binaaz-sale-project

**Goal:** Detect outliers in the numeric columns of `df_clean` (the version cleaned in Checkpoint 2), justify the choice of method, and process them appropriately.

This checkpoint builds on the previous step's output, so I first re-run the Checkpoint 2 null-handling steps (to recreate `df_clean`), then move on to the outlier analysis.

## 1. Rebuilding `df_clean` (from Checkpoint 2)

Quick recap: in Checkpoint 2 I dropped columns like `Binanın növü` (building type), `hour_y`, and `repair`; converted `vip`/`featured`/`mortgage`/`bill_of_sale` into boolean flags; created `seller_type`; recalculated `unit_price` from `price`/`Sahə`; and intentionally left structural gaps (`Mərtəbə`, `Torpaq sahəsi`) as NaN. Below I repeat those same steps.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 150)

df = pd.read_csv('house_sale.csv')

df_clean = df.copy()
df_clean = df_clean.drop(columns=["Binanın növü", "hour_y"])

df_clean["featured_flag"] = df_clean["featured"].notnull()
df_clean["vip_flag"] = df_clean["vip"].notnull()
df_clean["mortgage_flag"] = df_clean["mortgage"].notnull()
df_clean["bill_of_sale_flag"] = df_clean["bill_of_sale"].notnull() | df_clean["Çıxarış"].notnull()
df_clean = df_clean.drop(columns=["featured", "vip", "mortgage", "İpoteka", "bill_of_sale", "Çıxarış", "repair"])
df_clean["Təmir"] = df_clean["Təmir"].fillna("Unknown")

def seller_type(row):
    if pd.notnull(row["shop_name"]):
        return "Agency"
    elif pd.notnull(row["owner_name"]):
        return "Individual"
    return "Unknown"

df_clean["seller_type"] = df_clean.apply(seller_type, axis=1)
for col in ["shop_name", "shop_title", "owner_name", "owner_title"]:
    df_clean[col] = df_clean[col].fillna("N/A")
df_clean["products_label"] = df_clean["products_label"].fillna("Regular listing")
df_clean["description"] = df_clean["description"].fillna("")

def parse_area(val):
    if pd.isnull(val):
        return np.nan
    val = val.strip()
    if "sot" in val:
        return float(val.replace("sot", "").strip().replace(",", ".")) * 100
    return float(val.replace("m²", "").strip().replace(" ", ""))

df_clean["Sahə_numeric"] = df_clean["Sahə"].apply(parse_area)
df_clean["unit_price_calculated"] = (df_clean["price"] / df_clean["Sahə_numeric"]).round(2)
df_clean = df_clean.drop(columns=["unit_price"])

applicable_categories = ["Köhnə tikili", "Yeni tikili", "Ofis", "Həyət evi/Bağ evi"]
mask = df_clean["Kateqoriya"].isin(applicable_categories)
medians = df_clean.loc[mask].groupby("Kateqoriya")["Otaq sayı"].median()
for cat, med in medians.items():
    idx = df_clean[(df_clean["Kateqoriya"] == cat) & (df_clean["Otaq sayı"].isnull())].index
    df_clean.loc[idx, "Otaq sayı"] = med

print("df_clean is ready:", df_clean.shape)


df_clean is ready: (100775, 48)


## 2. Initial look at numeric column distributions

Before choosing an outlier-detection method, I look at the statistical summary and **skewness** of the key numeric columns (`price`, `Sahə_numeric`, `unit_price_calculated`, `views`, `Otaq sayı`). Skewness is decisive for the method choice: the z-score method assumes a normal distribution — if a distribution is heavily right-skewed, the mean and standard deviation that z-score relies on are themselves distorted by the extreme values, which can produce misleading results.

In [2]:
for col in ["price", "Sahə_numeric", "unit_price_calculated", "views", "Otaq sayı"]:
    print(f"--- {col} ---")
    print(df_clean[col].describe())
    print("skewness:", round(df_clean[col].skew(), 2))
    print()


--- price ---
count    1.007750e+05
mean     3.423557e+05
std      2.042627e+06
min      1.100000e+01
25%      1.450000e+05
50%      2.180000e+05
75%      3.380000e+05
max      6.000000e+08
Name: price, dtype: float64
skewness: 252.31

--- Sahə_numeric ---
count    1.007750e+05
mean     1.128061e+03
std      9.320227e+04
min      1.000000e+00
25%      7.000000e+01
50%      1.050000e+02
75%      1.600000e+02
max      2.000000e+07
Name: Sahə_numeric, dtype: float64
skewness: 152.35

--- unit_price_calculated ---
count    1.007750e+05
mean     2.321759e+03
std      1.042945e+04
min      0.000000e+00
25%      1.675000e+03
50%      2.214290e+03
75%      2.730500e+03
max      3.000000e+06
Name: unit_price_calculated, dtype: float64
skewness: 251.69

--- views ---
count    100775.000000
mean        700.065185
std        1680.672574
min          23.000000
25%          99.000000
50%         254.000000
75%         680.000000
max      113458.000000
Name: views, dtype: float64
skewness: 18.83

---

**Observation:** Every key column has a very high skewness value (`price`: 252, `Sahə_numeric`: 152, `unit_price_calculated`: 252, `views`: 19) — far from a normal distribution, strongly right-skewed. For example, the maximum `price` is 600,000,000 AZN, while the median is only 218,000 AZN.

**Method choice & justification:** For this reason, **I chose IQR (Interquartile Range) instead of z-score**. IQR is based on the median and quartiles, which are far less affected by extreme values than the mean/std that z-score relies on — making it a more reliable outlier-detection method for heavily skewed distributions.

## 3. A closer look at suspicious extreme values

Before applying IQR, I manually inspect the most extreme rows — because in some cases these aren't statistical outliers at all, just **data-entry errors** (e.g. an extra zero typed in by mistake).

In [3]:
print("--- Top 5 highest prices ---")
print(df_clean.nlargest(5, "price")[["price", "Kateqoriya", "city", "Sahə"]])
print()
print("--- Top 5 lowest prices ---")
print(df_clean.nsmallest(5, "price")[["price", "Kateqoriya", "city", "Sahə"]])


--- Top 5 highest prices ---
             price   Kateqoriya  city        Sahə
6605   600000000.0  Yeni tikili  bakı      200 m²
6942    60000000.0  Yeni tikili  bakı       50 m²
45825   40000000.0       Torpaq  bakı  200000 sot
72637   39000000.0       Torpaq  bakı     200 sot
8065    36000000.0       Obyekt  bakı     3000 m²

--- Top 5 lowest prices ---
       price         Kateqoriya  city    Sahə
38666   11.0             Obyekt  bakı   89 m²
22304   73.0        Yeni tikili  bakı   90 m²
39775  127.0  Həyət evi/Bağ evi  bakı  150 m²
21745  250.0  Həyət evi/Bağ evi  bakı   30 m²
11322  400.0        Yeni tikili  bakı   86 m²


**Observation:** The highest price — **600,000,000 AZN** for a 200 m² apartment in Baku — is far, far above the realistic market range for that size (typically 200,000–600,000 AZN) and is almost certainly a data-entry error (likely extra zeros). At the other extreme, there are prices like 11 AZN and 73 AZN — these also can't be real sale prices (likely a placeholder value meaning "negotiable").

I remove these **clearly implausible** values (impossible in the real property market) with a separate "sanity check" before running IQR — because these aren't statistical outliers, they're simply bad data.

In [4]:
print("Rows with price < 1000 AZN:", (df_clean["price"] < 1000).sum())

before = len(df_clean)
df_clean = df_clean[df_clean["price"] >= 1000].copy()
after = len(df_clean)
print(f"Rows removed by the sanity check: {before - after} ({round((before-after)/before*100,3)}% of the dataset)")


Rows with price < 1000 AZN: 10
Rows removed by the sanity check: 10 (0.01% of the dataset)


## 4. IQR-based outlier detection — grouped by category

A plain (global) IQR computes bounds across all rows at once, but in this dataset price scale depends heavily on `Kateqoriya` (e.g. the "normal" price range for `Torpaq` (land) is completely different from `Köhnə tikili` (old building)). So I compute the IQR bounds **separately for each `Kateqoriya`** — this way a "5-room office selling for 3 million" won't get wrongly excluded, while a "5-room apartment selling for 3 million" will correctly get flagged.

I compare the global and grouped approaches below to show the difference.

In [5]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

# Global IQR (for comparison)
low_g, high_g = iqr_bounds(df_clean["price"])
global_outliers = ((df_clean["price"] < low_g) | (df_clean["price"] > high_g)).sum()
print(f"Global IQR bounds: [{low_g:,.0f}, {high_g:,.0f}] -> {global_outliers} outliers ({global_outliers/len(df_clean)*100:.2f}%)")


Global IQR bounds: [-144,500, 627,500] -> 7322 outliers (7.27%)


In [6]:
def cap_grouped(df, col, group_col="Kateqoriya", k=1.5):
    """Computes IQR bounds per group, caps outliers at the bounds, and returns a flag column."""
    capped = df[col].copy()
    flag = pd.Series(False, index=df.index)
    bounds_report = {}
    for cat, group in df.groupby(group_col):
        low, high = iqr_bounds(group[col].dropna(), k)
        idx = group.index
        below = df.loc[idx, col] < low
        above = df.loc[idx, col] > high
        flag.loc[idx[below | above]] = True
        capped.loc[idx[below]] = low
        capped.loc[idx[above]] = high
        bounds_report[cat] = (round(low, 1), round(high, 1), int((below | above).sum()))
    return capped, flag, bounds_report

price_capped, price_flag, price_bounds = cap_grouped(df_clean, "price")
print("IQR bounds and outlier count per category (price):")
for cat, (low, high, n) in price_bounds.items():
    print(f"  {cat}: [{low:,.0f}, {high:,.0f}] -> {n} outliers")

print()
print(f"Total grouped outliers: {price_flag.sum()} ({price_flag.mean()*100:.2f}%)")
print(f"(Difference from global method: {global_outliers} -> {price_flag.sum()}, the grouped method is more accurate)")


IQR bounds and outlier count per category (price):
  Həyət evi/Bağ evi: [-342,500, 877,500] -> 1011 outliers
  Köhnə tikili: [22,500, 282,500] -> 815 outliers
  Obyekt: [-1,400,000, 3,000,000] -> 361 outliers
  Ofis: [-912,500, 2,307,500] -> 11 outliers
  Qaraj: [-17,500, 82,500] -> 3 outliers
  Torpaq: [-465,000, 903,000] -> 688 outliers
  Yeni tikili: [-72,500, 587,500] -> 2958 outliers

Total grouped outliers: 5847 (5.80%)
(Difference from global method: 7322 -> 5847, the grouped method is more accurate)


**Result:** The global IQR flags 7,322 rows (7.3%) as outliers, while the grouped IQR flags 5,847 rows (5.8%) — the difference comes from the global method incorrectly treating all "expensive" `Obyekt`/`Torpaq` listings as outliers, even though those prices are normal within their own category. For this reason, I'll use **category-grouped IQR** for all the remaining columns as well.

## 5. Applying to the remaining columns (`Sahə_numeric`, `unit_price_calculated`)

I apply the same grouped-IQR function to area (m²) and the calculated unit price too — since these also have a scale that depends on `Kateqoriya` (land plots are far larger than apartments).

In [7]:
area_capped, area_flag, area_bounds = cap_grouped(df_clean, "Sahə_numeric")
print("Sahə_numeric outlier count:", area_flag.sum(), f"({area_flag.mean()*100:.2f}%)")

unit_capped, unit_flag, unit_bounds = cap_grouped(df_clean, "unit_price_calculated")
print("unit_price_calculated outlier count:", unit_flag.sum(), f"({unit_flag.mean()*100:.2f}%)")


Sahə_numeric outlier count: 4077 (4.05%)
unit_price_calculated outlier count: 3352 (3.33%)


## 6. The `views` column — global IQR

`views` (view count) doesn't depend on `Kateqoriya` — it's a metadata field reflecting a listing's popularity, driven by factors like how long it's been active rather than the property type. So a **global IQR** is sufficient here, no grouping needed.

In [8]:
low_v, high_v = iqr_bounds(df_clean["views"])
views_flag = (df_clean["views"] < low_v) | (df_clean["views"] > high_v)
views_capped = df_clean["views"].clip(low_v, high_v)
print(f"views IQR bounds: [{low_v:,.0f}, {high_v:,.0f}] -> {views_flag.sum()} outliers ({views_flag.mean()*100:.2f}%)")


views IQR bounds: [-772, 1,552] -> 10441 outliers (10.36%)


## 7. Decision: remove, or cap?

There were two main options:

1. **Remove outlier rows entirely** — simple, but means losing 5–10% of the data, and **not all of these outliers are errors** (e.g. luxury/high-end listings could be genuinely real, just rare within their category).
2. **Capping (winsorizing)** — pull the outlier value down/up to the boundary (Q1-1.5×IQR / Q3+1.5×IQR). Keeps the row, but limits its statistical influence (e.g. distorting the mean).

**Decision: I chose capping, not deletion**, because:
- Obvious data-entry errors (like the 600M AZN listing) were already removed separately in the sanity check in step 3.
- Most of the remaining outliers could well be genuine (luxury properties, large land plots) — deleting them would mean losing the entire "high-end segment" of the dataset, which could matter for future analysis/modeling.
- I also keep an `is_outlier_*` flag column for each field so it's always visible who was flagged as an outlier — meaning the capped value isn't mandatory to use; it's possible to go back to the original if needed.

Applying this decision below.

In [9]:
df_clean["price_capped"] = price_capped
df_clean["is_outlier_price"] = price_flag

df_clean["Sahə_numeric_capped"] = area_capped
df_clean["is_outlier_area"] = area_flag

df_clean["unit_price_capped"] = unit_capped
df_clean["is_outlier_unit_price"] = unit_flag

df_clean["views_capped"] = views_capped
df_clean["is_outlier_views"] = views_flag

print(df_clean[["price", "price_capped", "is_outlier_price"]].describe())


              price  price_capped
count  1.007650e+05  1.007650e+05
mean   3.423897e+05  2.834652e+05
std    2.042725e+06  2.835427e+05
min    1.000000e+03  1.000000e+03
25%    1.450000e+05  1.450000e+05
50%    2.180000e+05  2.180000e+05
75%    3.380000e+05  3.300000e+05
max    6.000000e+08  3.000000e+06


## 8. Final check — before/after comparison

Showing capping's effect on the mean and standard deviation — the goal was to reduce the statistical distortion caused by extreme values.

In [10]:
compare = pd.DataFrame({
    "price (before)": df_clean["price"].describe(),
    "price_capped (after)": df_clean["price_capped"].describe(),
})
compare


,price (before),price_capped (after)
count,1.007650e+05,1.007650e+05
mean,3.423897e+05,2.834652e+05
std,2.042725e+06,2.835427e+05
min,1.000000e+03,1.000000e+03
25%,1.450000e+05,1.450000e+05
50%,2.180000e+05,2.180000e+05
75%,3.380000e+05,3.300000e+05
max,6.000000e+08,3.000000e+06


In [11]:
print("Row count unchanged (only capping was applied):", len(df_clean))
print()
print("Outlier percentage per column:")
for col in ["is_outlier_price", "is_outlier_area", "is_outlier_unit_price", "is_outlier_views"]:
    print(f"  {col}: {df_clean[col].mean()*100:.2f}%")


Row count unchanged (only capping was applied): 100765

Outlier percentage per column:
  is_outlier_price: 5.80%
  is_outlier_area: 4.05%
  is_outlier_unit_price: 3.33%
  is_outlier_views: 10.36%


## Checkpoint 3 — Summary

- Since the key numeric columns (`price`, `Sahə_numeric`, `unit_price_calculated`) had very high skewness (150–250 range), I chose **IQR instead of z-score** — z-score assumes a normal distribution, and since its mean/std are themselves distorted by the extreme values, it would be misleading for this dataset.
- Before applying IQR, I removed obvious errors that are impossible in the real market (e.g. a 600 million AZN apartment, an 11 AZN office) with a separate sanity check (10 rows, ~0.01%).
- Applied IQR **grouped by `Kateqoriya`** rather than globally — since price scale depends on property type; this produced a more accurate result than the global method (5.8% outliers instead of 7.3%).
- Chose to **cap outliers rather than delete them** (winsorizing at the Q1-1.5×IQR / Q3+1.5×IQR bounds) — because many of these outliers could be genuine (luxury properties), and deleting them risked losing a valuable segment of the dataset. Kept an `is_outlier_*` flag for each column for transparency.
- Row count stayed essentially unchanged (only 10 rows removed in the sanity check) — this brought the `price` mean down from 342,390 AZN to 283,465 AZN, which aligns much better with the real median (218,000 AZN).

